# Семинар 2. Питон для работы с моделью

**Тема:** базовый питон, без которого не поговорить с моделью — переменные, списки, словари и функции. Учимся читать ответ модели и доставать из него нужное.

На прошлом семинаре мы отправляли запросы и меняли промпты. Сегодня заглянем чуть глубже: **как устроен питон**, на котором всё это работает.

Как и в прошлый раз — **сначала запускаем ячейку, потом смотрим результат, потом меняем одно значение и запускаем снова.**

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sblenlkj/summer-agent-school-seminars-1-2/blob/main/notebooks/seminar_2_8_9.ipynb)


## 1. План

1. **переменные** — как хранить текст и числа;
2. **f-строки** — как собирать запрос из кусочков;
3. **списки** — как хранить много значений и обходить их циклом;
4. **словари** — как хранить данные парами «ключ — значение» (и почему сообщение модели — это словарь);
5. как **прочитать ответ модели** и достать из него нужное;
6. **функции** — как не повторять один и тот же код.

И всё это — на живом примере: мы постоянно будем что-то спрашивать у модели.


## 2. Настройка — просто запусти

Эти две ячейки технические — они подключают модель (как на прошлом семинаре). **Просто запустите их и идём дальше.** Вставьте ключ, когда попросят.


In [ ]:
import uuid
import requests
import warnings
from urllib3.exceptions import InsecureRequestWarning

warnings.filterwarnings("ignore", category=InsecureRequestWarning)

AUTH_URL = "https://ngw.devices.sberbank.ru:9443/api/v2/oauth"
BASE_URL = "https://gigachat.devices.sberbank.ru/api/v1/chat/completions"
MODEL_NAME = "GigaChat-2"

AUTH_KEY = input("Вставьте ключ авторизации GigaChat и нажмите Enter: ")


def get_access_token(auth_key):
    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Accept": "application/json",
        "RqUID": str(uuid.uuid4()),
        "Authorization": f"Basic {auth_key}",
    }
    response = requests.post(AUTH_URL, headers=headers, data={"scope": "GIGACHAT_API_PERS"}, verify=False)
    print("Статус ответа:", response.status_code)
    if response.status_code != 200:
        print(response.text)
        return None
    return response.json()["access_token"]


ACCESS_TOKEN = get_access_token(AUTH_KEY)
print("Токен получен!" if ACCESS_TOKEN else "Токен не получен, проверьте ключ.")

In [ ]:
# Функция ask_model — отправляет вопрос модели и возвращает текст ответа.
# Сегодня мы поймём, как она устроена внутри. Пока просто запустите.
def ask_model(prompt, temperature=0.7, max_tokens=512):
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    headers = {
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    response = requests.post(BASE_URL, headers=headers, json=payload, verify=False)
    if response.status_code != 200:
        print("Ошибка запроса:", response.status_code)
        return None
    return response.json()["choices"][0]["message"]["content"]


print(ask_model("Скажи коротко: всё работает?"))

## 3. Переменные

**Переменная** — это коробочка с именем, в которой что-то лежит. В неё можно положить текст или число, а потом обращаться к ней по имени.

Запустите ячейку и посмотрите, что в коробочках.


In [ ]:
name = "Дима"      # текст (его берут в кавычки) — это строка
age = 14            # число (без кавычек)
school_subject = "информатика"

print(name)
print(age)
print("Любимый предмет:", school_subject)

Главная польза переменной: положили один раз — используем много раз.

Положим в переменную **целый промпт**, а потом отправим его модели.


In [ ]:
my_prompt = "Объясни простыми словами, что такое переменная в программировании."

print("Мы отправим модели такой запрос:")
print(my_prompt)
print()
print("Ответ модели:")
print(ask_model(my_prompt))

## 4. f-строки: собираем запрос из кусочков

Часто нужно собрать текст из нескольких переменных. Для этого в питоне есть **f-строка**: ставим букву `f` перед кавычками, а переменные пишем в `{фигурных скобках}`.

Посмотрите: мы складываем `name` и `topic` прямо внутри текста.


In [ ]:
name = "Аня"
topic = "что такое токен"

prompt = f"Меня зовут {name}. Объясни мне, {topic}, простыми словами и коротко."

print("Получился такой промпт:")
print(prompt)
print()
print(ask_model(prompt))

### Мини-задание 1

Поменяйте значения переменных `name` и `topic` ниже (только текст в кавычках) и запустите. Промпт соберётся сам.


In [ ]:
name = "TODO ваше имя"
topic = "TODO про что спросить, например: что такое промпт"

prompt = f"Меня зовут {name}. Объясни мне, {topic}, простыми словами и коротко."
print(ask_model(prompt))

## 5. Списки

**Список** — это коробка, в которой лежит сразу **много значений по порядку**. Список пишут в квадратных скобках `[ ]`.

Запустите и посмотрите, как доставать элементы и считать их количество.


In [ ]:
terms = ["переменная", "список", "словарь", "функция"]

print("Весь список:", terms)
print("Первый элемент:", terms[0])   # счёт идёт с нуля!
print("Второй элемент:", terms[1])
print("Сколько всего:", len(terms))

Самое удобное в списке — его можно **обойти циклом** `for`: взять каждый элемент по очереди.

Сейчас для **каждого** слова из списка мы попросим у модели короткое объяснение.


In [ ]:
terms = ["переменная", "список", "словарь"]

for term in terms:
    prompt = f"Объясни термин «{term}» одним предложением, для новичка."
    print("=" * 60)
    print("Слово:", term)
    print(ask_model(prompt))

### Мини-задание 2

Добавьте в список `terms` ещё 1-2 слова (в кавычках, через запятую) и запустите. Цикл сам спросит про каждое.


In [ ]:
terms = ["цикл", "функция"]   # TODO: добавьте свои слова

for term in terms:
    prompt = f"Объясни термин «{term}» одним предложением, для новичка."
    print("=" * 60)
    print("Слово:", term)
    print(ask_model(prompt))

## 6. Словари

**Словарь** хранит данные парами **«ключ — значение»**. Пишут в фигурных скобках `{ }`. Это как настоящий словарь: по слову (ключу) находим перевод (значение).

Запустите и посмотрите, как достать значение по ключу.


In [ ]:
student = {
    "name": "Дима",
    "age": 14,
    "city": "Москва",
}

print("Весь словарь:", student)
print("Имя:", student["name"])    # достаём значение по ключу
print("Возраст:", student["age"])

### Мостик к промптам: сообщение модели — это тоже словарь

Помните, на прошлом семинаре в запросе была вот такая строчка?

```python
{"role": "user", "content": "Привет!"}
```

Это **словарь** с двумя ключами:

- `role` — **роль**, кто говорит (`user` — пользователь, то есть мы; `system` — инструкция модели; `assistant` — ответ модели);
- `content` — сам **текст** сообщения.

То есть «роль» и «инструкция», про которые шла речь на лекции про промпты, — это просто ключи в обычном словаре. Запустите ячейку.


In [ ]:
message = {
    "role": "user",
    "content": "Сколько будет 2 + 2?",
}

print("Роль:", message["role"])
print("Текст:", message["content"])

## 7. Список словарей = диалог с моделью

Теперь соединим списки и словари. Если положить **несколько сообщений-словарей в список**, получится `messages` — целый диалог.

- `system` — задаёт, **как** модель себя ведёт;
- `user` — наш вопрос.

Запустите. Функция `ask_chat` отправляет такой список сообщений модели.


In [ ]:
def ask_chat(messages, temperature=0.7, max_tokens=512):
    payload = {"model": MODEL_NAME, "messages": messages,
               "temperature": temperature, "max_tokens": max_tokens}
    headers = {
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    response = requests.post(BASE_URL, headers=headers, json=payload, verify=False)
    if response.status_code != 200:
        print("Ошибка запроса:", response.status_code)
        return None
    return response.json()["choices"][0]["message"]["content"]


messages = [
    {"role": "system", "content": "Ты весёлый учитель. Отвечай очень коротко и с примером."},
    {"role": "user", "content": "Что такое список в программировании?"},
]

print(ask_chat(messages))

### Мини-задание 3

Поменяйте **только текст** в сообщении с ролью `system` (например: «Ты строгий профессор» или «Ты пират») и запустите. Посмотрите, как изменится стиль ответа — а вопрос остался тот же.


In [ ]:
messages = [
    {"role": "system", "content": "TODO опишите, как модель должна себя вести"},
    {"role": "user", "content": "Что такое список в программировании?"},
]

print(ask_chat(messages))

## 8. Читаем ответ модели

Теперь самое важное умение семинара — **достать из ответа модели именно то, что нужно**.

Когда модель отвечает, она присылает не просто текст, а **JSON** — это вложенные друг в друга **словари и списки**. То есть всё то, что мы уже знаем!

Сначала посмотрим на «сырой» ответ.


In [ ]:
payload = {
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "Назови один интересный факт о космосе."}],
}
headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}

response = requests.post(BASE_URL, headers=headers, json=payload, verify=False)
data = response.json()   # превращаем ответ в словари и списки

print("Это словарь. Вот его ключи:")
print(data.keys())

Внутри `data` лежит ключ `choices` — это **список** вариантов ответа. Берём первый вариант, в нём словарь `message`, а в нём ключ `content` — наш текст.

Давайте достанем текст **по шагам**, чтобы увидеть, как устроен «адрес».


In [ ]:
choices = data["choices"]          # это список вариантов ответа
first = choices[0]                  # берём первый вариант (счёт с нуля)
message = first["message"]          # внутри — словарь сообщения
text = message["content"]           # а в нём — сам текст

print("Текст ответа:")
print(text)
print()
print("Сколько токенов потрачено:", data["usage"])

Те же самые шаги можно записать **в одну строчку** — это и есть та «длинная» строка из прошлого семинара:

```python
data["choices"][0]["message"]["content"]
```

Теперь понятно, что это просто: зайди в словарь `data` → в список `choices` → возьми первый элемент → в нём словарь `message` → достань ключ `content`.


In [ ]:
# Та же самая запись, но в одну строку:
answer = data["choices"][0]["message"]["content"]
print(answer)

## 9. Функции

Мы уже несколько раз писали почти одинаковый код: собрать запрос → отправить → достать текст. Чтобы **не повторяться**, такой код прячут в **функцию**.

Функция — это как своя личная команда. У неё есть:

- имя (например, `my_ask`);
- то, что ей передают (вопрос);
- то, что она возвращает (ответ).

Запустите — ячейка готовит функцию.


In [ ]:
def my_ask(question):
    messages = [{"role": "user", "content": question}]
    answer = ask_chat(messages)
    return answer


print("Функция my_ask готова.")

Теперь её можно вызывать сколько угодно раз — коротко и удобно. Именно так и устроена `ask_model`, которой мы пользовались.


In [ ]:
print(my_ask("Объясни, зачем нужны функции в программировании. Коротко."))
print("-" * 60)
print(my_ask("Приведи смешной пример из жизни про функции."))

## 10. Что мы сегодня сделали

Мы разобрали базовый питон **на живом примере модели**:

- **переменные** — хранят текст и числа;
- **f-строки** — собирают промпт из кусочков;
- **списки** — хранят много значений, их удобно обходить циклом `for`;
- **словари** — хранят пары «ключ — значение»; одно сообщение модели — это словарь `{role, content}`;
- список словарей — это `messages`, целый диалог;
- ответ модели — это вложенные словари и списки, и мы умеем доставать из них текст;
- **функции** — прячут повторяющийся код.

Главная мысль:

> Чтобы говорить с моделью, не нужно магии. Запрос и ответ — это обычные переменные, списки и словари. Зная их, мы понимаем, как всё устроено внутри.


## 11. Домашнее задание (по желанию)

1. Создайте свой словарь `student` с ключами `name`, `age`, `hobby` и распечатайте каждое значение по ключу.
2. Сделайте список из 3 своих вопросов и спросите их у модели циклом `for`.
3. Напишите свою функцию `ask_funny(question)`, которая добавляет к вопросу «ответь смешно» и возвращает ответ модели.

И, как всегда, — **запускайте и смотрите**, что получается.
